# 中等教育数据宽表构建全流程
_version: 2026-08-20_

In [ ]:
import pandas as pd
import numpy as np

## 1 - 构建头部学校名单

### 1.1 全国排名

#### 1.1.1 出国留学网-全国高中排名100强

In [ ]:
# 读取出国留学网-全国高中排名100强名单
df_top100 = pd.read_csv('03-中间结果/出国留学网-全国高中排名100强.csv')
df_top100.tail()

In [ ]:
# 剔除重复数据
df_top100.drop_duplicates('学校名称', keep='first', inplace=True)

# 保留有效字段
df_top100 = df_top100[['学校名称', '属性', '性质', '类型', '学校地址', 
                       '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_top100.tail()

#### 1.1.2 个人图书馆-全国高中排名名单

In [ ]:
# 读取个人图书馆-全国高中排名名单
df_high_rank = pd.read_csv('03-中间结果/个人图书馆-全国高中排名名单.csv')
df_high_rank.head()

In [ ]:
def normalize_province(value):
    """将省级行政区名称归一化，用于阻止跨省学校误匹配。"""
    if pd.isna(value):
        return None

    province = str(value).strip()
    for suffix in ('特别行政区', '壮族自治区', '回族自治区', '维吾尔自治区', '自治区', '省', '市'):
        if province.endswith(suffix):
            return province[:-len(suffix)]
    return province


# 排名源中的省份必须与基本信息匹配结果一致；跨省结果是错误实体，不得进入宽表。
source_province = df_high_rank['省市'].map(normalize_province)
matched_province = df_high_rank['省级位置'].map(normalize_province)
province_mismatch = source_province.notna() & matched_province.notna() & source_province.ne(matched_province)
invalid_match_cols = ['数据源名称', '地区', '属性', '性质', '学校地址', '联系电话', '地级位置', '县级位置']
df_high_rank.loc[province_mismatch, invalid_match_cols] = np.nan
df_high_rank.loc[province_mismatch, '类型'] = '中学'
df_high_rank.loc[province_mismatch, '省级位置'] = df_high_rank.loc[province_mismatch, '省市']
print("已剔除跨省学校误匹配数量: ", province_mismatch.sum())

# 保留有效字段
df_high_rank = df_high_rank[['学校名称', '属性', '性质', '类型', '学校地址',
                             '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_high_rank.tail()

#### 1.1.3 买购网-中国十大国际高中学校排名

In [ ]:
# 读取买购网-中国十大国际高中学校排名
df_inter_top10 = pd.read_csv('03-中间结果/买购网-中国十大国际高中学校排名.csv')
df_inter_top10.tail()

In [ ]:
# 调整列名
df_inter_top10.rename(columns={'学校名': '学校名称'}, inplace=True)

# 保留有效字段
df_inter_top10 = df_inter_top10[['学校名称', '属性', '性质', '类型', '学校地址', 
                                 '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_inter_top10.tail()

#### 1.1.4 买购网-全国16所能保送的外国语学校排行榜

In [ ]:
# 读取买购网-全国16所能保送的外国语学校排行榜
df_foreign_16 = pd.read_csv('03-中间结果/买购网-全国16所能保送的外国语学校排行榜.csv')
df_foreign_16.tail()

In [ ]:
# 调整列名
df_foreign_16.rename(columns={'学校名': '学校名称'}, inplace=True)

# 保留有效字段
df_foreign_16 = df_foreign_16[['学校名称', '属性', '性质', '类型', '学校地址', 
                               '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_foreign_16.tail()

### 1.2 各省排名

#### 1.2.1 买购网-各省top10中学名单

In [ ]:
# 读取买购网-各省top10中学名单
df_prov_top10 = pd.read_csv('03-中间结果/买购网-各省top10中学名单.csv')
df_prov_top10.tail()

In [ ]:
# 调整列名
df_prov_top10.rename(columns={'学校名': '学校名称'}, inplace=True)

# 剔除重复数据
df_prov_top10.drop_duplicates('学校名称', keep='first', inplace=True)

# 保留有效字段
df_prov_top10 = df_prov_top10[['学校名称', '属性', '性质', '类型', '学校地址', 
                               '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_prov_top10.tail()

#### 1.2.2 中职招生网-全国31省市高中TOP5名单

In [ ]:
# 读取中职招生网-全国31省市高中TOP5名单
df_prov_top5 = pd.read_csv('03-中间结果/中职招生网-全国31省市高中TOP5名单.csv')
df_prov_top5.head()

In [ ]:
# 调整列名
df_prov_top5.rename(columns={'学校名': '学校名称'}, inplace=True)

# 保留有效字段
df_prov_top5 = df_prov_top5[['学校名称', '属性', '性质', '类型', '学校地址', 
                               '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_prov_top5.tail()

#### 1.2.3 郁金香-全国各省市地区重点高中前十排名 

In [ ]:
# 读取郁金香-全国各省市地区重点高中前十排名
df_yjx = pd.read_csv('03-中间结果/郁金香-全国各省市地区重点高中前十排名.csv')
df_yjx.head()

In [ ]:
# 保留有效字段
df_yjx = df_yjx[['学校名称', '属性', '性质', '类型', '学校地址', 
                 '联系电话', '省级位置', '地级位置', '县级位置']].copy().reset_index(drop=True)
df_yjx.tail()

### 1.3 头部学校名单合并

In [ ]:
# 合并头部学校名单
df_top_school = pd.concat([df_top100, df_prov_top10, df_inter_top10, df_foreign_16, 
                           df_high_rank, df_prov_top5, df_yjx], ignore_index=True)
df_top_school.tail()

In [ ]:
# 剔除完全相同的重复数据
df_top_school.drop_duplicates(keep='first', inplace=True)
df_top_school = df_top_school.reset_index(drop=True)
df_top_school.tail()

In [ ]:
# 学校名称重复的例子
df_top_school[df_top_school.duplicated('学校名称', keep=False)].sort_values('学校名称')

In [ ]:
# 删除学校名重复的数据
df_top_school.drop_duplicates('学校名称', keep='first', inplace=True)
df_top_school = df_top_school.reset_index(drop=True)
df_top_school.tail()

In [ ]:
# 保存头部学校名单
df_top_school.to_csv('03-中间结果/头部学校名单.csv', index=False)
df_top_school.head()

## 2 - 更多学校和综合信息补充

### 2.1 我要搜学网-中学

In [ ]:
# 读取我要搜学网的中学信息表
df_souxue_norm = pd.read_csv('01-数据源/03-综合信息/我要搜学网-中学信息表.csv')
df_souxue_norm.rename(columns={'中学名称': '学校名称'}, inplace=True)
df_souxue_norm = df_souxue_norm.loc[:, ['学校名称', '属性', '性质', '类型', '学校地址', 
                                        '联系电话', '省级位置', '地级位置', '县级位置']]
df_souxue_norm.head()

### 2.2 我要搜学网-职业中学

In [ ]:
# 读取我要搜学网的职业学校信息表
df_souxue_prof = pd.read_csv('01-数据源/03-综合信息/我要搜学网-职业学校信息表.csv')
df_souxue_prof.rename(columns={'中学名称': '学校名称'}, inplace=True)
df_souxue_prof = df_souxue_prof.loc[:, ['学校名称', '属性', '性质', '类型', '学校地址', 
                                        '联系电话', '省级位置', '地级位置', '县级位置']]
df_souxue_prof.tail()

In [ ]:
# 剔除职业学校中属于高校的学校
df_college = pd.read_csv('../数据收集-高等教育/04-数据宽表/高等教育数据宽表v20230103.csv')
df_souxue_prof = df_souxue_prof[~df_souxue_prof['学校名称'].isin(df_college['学校名称'].values)].copy()
df_souxue_prof = df_souxue_prof.reset_index(drop=True)
df_souxue_prof.tail()

In [ ]:
# 保留中职类型的学校
df_souxue_prof = df_souxue_prof[df_souxue_prof['学校名称'].str.contains('中专|高中|中学')].reset_index(drop=True)
df_souxue_prof.tail()

### 2.3 我要搜学网-民办中学

In [ ]:
# 读取我要搜学网-民办中学信息表
df_souxue_mb = pd.read_excel('01-数据源/03-综合信息/我要搜学网-民办中学.xlsx', skiprows=1)
df_souxue_mb.rename(columns={'办学性质': '性质'}, inplace=True)
df_souxue_mb['省级位置'] = df_souxue_mb['地区'].apply(lambda x: x.split(' ')[0])
df_souxue_mb['地级位置'] = df_souxue_mb['地区'].apply(lambda x: x.split(' ')[1] if len(x.split(' '))==3 else x.split(' ')[0])
df_souxue_mb['县级位置'] = df_souxue_mb['地区'].apply(lambda x: x.split(' ')[-1])
df_souxue_mb.drop(['序号', '地区'], axis=1, inplace=True)
df_souxue_mb.tail()

In [ ]:
# 合并中学,中职学校,民办中学
df_souxue = pd.concat([df_souxue_norm, df_souxue_prof, df_souxue_mb], ignore_index=True)
df_souxue.drop_duplicates('学校名称', keep='last', inplace=True)
df_souxue = df_souxue.reset_index(drop=True)
df_souxue.tail()

### 2.4 学校大全-全国初中和高中信息

In [ ]:
# 读取全国初中信息表
df_middle = pd.read_csv('01-数据源/03-综合信息/学校大全-全国初中信息表.csv')

# 修正列名
df_middle.columns = ['学校名称', '学校地址', '联系电话', '官网', '城市']
df_middle.tail()

In [ ]:
# 读取全国高中信息表
df_high = pd.read_csv('01-数据源/03-综合信息/学校大全-全国高中信息表.csv')

# 修正列名
df_high.columns = ['学校名称', '学校地址', '联系电话', '官网', '城市']
df_high.tail()

In [ ]:
# 合并初高中学校
df_web = pd.concat([df_middle, df_high], ignore_index=True)
df_web.tail()

In [ ]:
# 删除重复信息
df_web.drop_duplicates(keep='first', inplace=True)
df_web = df_web.reset_index(drop=True)
df_web.tail()

In [ ]:
# 保留有效的官网信息
df_web = df_web[['学校名称', '官网']].copy()
df_web.dropna(subset=['官网'], axis=0, inplace=True)
df_web = df_web.reset_index(drop=True)
df_web.tail()

In [ ]:
# 网站信息重复的案例
df_web[df_web.duplicated('学校名称', keep=False)].sort_values('学校名称')
df_web[df_web.duplicated('学校名称', keep=False)].sort_values('学校名称').to_csv('03-中间结果/学校大全-网站信息重复.csv', index=False)

In [ ]:
# 剔除网站信息重复的数据
df_web.drop_duplicates('学校名称', keep='last', inplace=True)

In [ ]:
# 与头部学校名单合并
df_top_school = df_top_school.merge(df_web, how='left', on='学校名称')
print("匹配到的学校官网数量: ", len(df_top_school[~df_top_school['官网'].isna()]))
df_top_school.tail()

In [ ]:
# 与我要搜学网的信息合并
df_info = df_souxue.merge(df_web, how='left', on='学校名称')
print("匹配到的学校官网数量: ", len(df_info[~df_info['官网'].isna()]))
df_info.tail()

### 2.5 信息合并

In [ ]:
# 从补充学校中排除头部学校
df_plus = df_info[~df_info['学校名称'].isin(df_top_school['学校名称'])].copy()

# 将头部学校与补充学校合并
df_wide = pd.concat([df_top_school, df_plus], ignore_index=True)
df_wide.head()

## 3 - 地理信息

In [ ]:
# 读取高德地图查询结果
df_geo = pd.read_csv('01-数据源/04-地理信息/全量中学经纬度补全_合并原表和经纬度表_with匹配度.csv')
df_geo.head()

In [ ]:
# 提取经纬度信息；头部学校的专项核验结果优先于全量模糊匹配结果。
df_geo_head = pd.read_csv('01-数据源/04-地理信息/头部中学经纬度数据补全_1227.csv')
head_geo_info = df_geo_head[['查询', '省份', '经度', '纬度']].copy()
head_geo_info.rename(columns={'查询': '学校名称'}, inplace=True)
full_geo_info = df_geo[['查询', '省份', '经度', '纬度']].copy()
full_geo_info.rename(columns={'查询': '学校名称'}, inplace=True)

geo_info = pd.concat([head_geo_info, full_geo_info], ignore_index=True)
geo_info.drop_duplicates('学校名称', keep='first', inplace=True)
geo_info.rename(columns={'省份': '地理匹配省份'}, inplace=True)
geo_info.head()

In [ ]:
# 将经纬度信息合并至中等教育宽表
df_wide = df_wide.merge(geo_info, on='学校名称', how='left')
df_wide.head()

## 4 - 人工数据补全

### 4.1 学校官网数据补全（2022-12-20）

In [ ]:
# 读取补全数据表
df_fill_web = pd.read_csv('05-补全计划/中学官网数据补全v20221220.csv')

# 重复学校保留首个记录
df_fill_web.drop_duplicates('学校名称', keep='first', inplace=True)
df_fill_web.head()

In [ ]:
# 人工核验表是头部学校的权威修订源，需同步地址、电话和行政区划，而非只更新官网。
fill_cols = ['属性', '性质', '类型', '学校地址', '联系电话',
             '省级位置', '地级位置', '县级位置', '官网']
wide_by_name = df_wide.set_index('学校名称')
fill_by_name = df_fill_web.set_index('学校名称')
overlap_names = wide_by_name.index.intersection(fill_by_name.index)
wide_by_name.loc[overlap_names, fill_cols] = fill_by_name.loc[overlap_names, fill_cols]
df_wide = wide_by_name.reset_index()

# 人工修订应用后再次核验地理命中；跨省坐标宁可留空，也不能关联到另一所同名学校。
record_province = df_wide['省级位置'].map(normalize_province)
matched_geo_province = df_wide['地理匹配省份'].map(normalize_province)
geo_province_mismatch = (record_province.notna() & matched_geo_province.notna()
                         & record_province.ne(matched_geo_province))
df_wide.loc[geo_province_mismatch, ['经度', '纬度']] = np.nan
print("已剔除跨省地理误匹配数量: ", geo_province_mismatch.sum())
df_wide.drop(columns='地理匹配省份', inplace=True)

## 5 - 中等教育宽表

### 5.1 中学数据宽表

In [ ]:
# 保存中学宽表结果
df_wide_norm = df_wide[df_wide['类型'] == '中学'].copy().reset_index(drop=True)

# Issue #1 及同源误匹配的回归检查。
expected_corrections = {
    '六盘水市第三中学': ('贵州省', '六盘水市', '钟山区', 104.825447, 26.59926),
    '阜阳市第三中学': ('安徽省', '阜阳市', '颍州区', 115.793036, 32.865671),
    '福建省漳平市第三中学': ('福建省', '龙岩市', '漳平市', 117.416908, 25.315541),
}
for school_name, expected in expected_corrections.items():
    school_rows = df_wide_norm[df_wide_norm['学校名称'] == school_name]
    assert len(school_rows) == 1, f'{school_name} 应且只能保留一条记录'
    actual = school_rows.iloc[0]
    actual_values = (actual['省级位置'], actual['地级位置'], actual['县级位置'],
                     actual['经度'], actual['纬度'])
    assert actual_values == expected, f'{school_name} 的行政区划或坐标发生回归: {actual_values}'

df_wide_norm.to_csv('04-数据宽表/中等教育数据宽表-中学.csv', index=False)
df_wide_norm.tail()

### 5.2 中职学校数据宽表

In [ ]:
# 读取职教网-中职学校信息表
df_1 = pd.read_csv('01-数据源/03-综合信息/职教网_中职学校信息表1.csv')
df_2 = pd.read_csv('01-数据源/03-综合信息/职教网_中职学校信息表2.csv')
df_voc = pd.concat([df_1, df_2], ignore_index=True)

# 剔除学校名称为空的无效信息
df_voc.dropna(subset=['学校名称'], axis=0, inplace=True)
df_voc = df_voc.reset_index(drop=True)

# 剔除无效字段
df_voc.drop(['网页id', '其他可补充字段', '电子简章'], axis=1, inplace=True)

# 调整字段名称
df_voc.rename(columns={'职业': '开设专业'}, inplace=True)

# 过滤学校名称有误的数据
bad_names = df_voc['学校名称'].str.startswith('<img')
df_voc = df_voc[~bad_names].copy().reset_index(drop=True)
df_voc.tail()

In [ ]:
# 保存中职学校数据宽表
df_voc.to_csv('04-数据宽表/中等教育数据宽表-中职学校.csv', index=False)

## 6 - 数据补全计划

TBD